# Phase 3 — dbt Transformation & Data Quality

## What is dbt and why do we use it here?

**dbt (Data Build Tool)** is the standard tool for the **Transform** step in an ELT pipeline.  
Think of it as a framework that wraps your SQL models with:

| Feature | What it does |
|---------|-------------|
| **Dependency graph** | `{{ ref('stg_match_results') }}` automatically runs staging before marts |
| **Materialization** | Decides if a model is a `VIEW` (fast, no storage) or `TABLE` (persisted) |
| **Data quality tests** | `not_null`, `unique`, `accepted_values` run with `dbt test` |
| **Documentation** | `dbt docs generate` builds a searchable data catalog |
| **Lineage** | Visual DAG showing which models depend on which |

**Where does dbt fit in our pipeline?**
```
PySpark (clean raw CSVs → Parquet + raw Postgres tables)
    ↓
dbt (SQL transformations: raw → staging → marts)
    ↓
mart_predictions_input (ready for ML model in Phase 4)
```

## Models we build

```
SOURCES (raw tables from Phase 1)
    └── raw_results

STAGING (views — lightweight cleaning)
    ├── stg_match_results     Clean types, add result column
    ├── stg_rankings          Derived strength scores
    └── stg_fixtures          2026 WC group stage fixtures

MARTS (tables — persisted aggregations)
    ├── mart_team_form        Last 10/20 match stats per team
    ├── mart_head_to_head     All-time H2H records
    └── mart_predictions_input  ML-ready feature table
```

## Prerequisites
- Phase 1 notebook ran (raw tables in PostgreSQL)
- Docker Postgres is running: `docker-compose up postgres -d`
- dbt installed: `pip install dbt-postgres==1.8.3`
- `.env` file exists with `POSTGRES_*` variables

In [ ]:
# ============================================================
# Cell 2 — Run dbt commands via subprocess
# ============================================================
# We use Python's subprocess module to call dbt from the notebook.
# This is the same as typing these commands in a terminal,
# but it lets us capture and display output inline.

import subprocess
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Load .env so dbt can read env_var() references in profiles.yml
load_dotenv(dotenv_path=Path("..") / ".env")

# Path to the dbt project (one level up from notebooks/)
DBT_PROJECT_DIR = str(Path("..") / "dbt")

def run_dbt(command: str, capture: bool = True) -> tuple[int, str, str]:
    """
    Run a dbt command in the dbt project directory.
    
    Returns: (return_code, stdout, stderr)
    
    --profiles-dir . tells dbt to look for profiles.yml in the
    current directory (dbt/) instead of the default ~/.dbt/
    """
    full_command = f"dbt {command} --profiles-dir ."
    print(f"\n{'='*60}")
    print(f"Running: {full_command}")
    print(f"Directory: {DBT_PROJECT_DIR}")
    print('='*60)
    
    result = subprocess.run(
        full_command,
        shell=True,
        cwd=DBT_PROJECT_DIR,
        capture_output=capture,
        text=True,
        env=os.environ.copy()  # Pass all env vars including those from .env
    )
    
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    
    status = "SUCCESS" if result.returncode == 0 else "FAILED"
    print(f"\n[{status}] Return code: {result.returncode}")
    
    return result.returncode, result.stdout, result.stderr

# ---- Step 1: Install dbt dependencies ----
# dbt_project.yml might reference dbt packages — install them first
print("Step 1: dbt deps (install any dbt packages)")
rc, out, err = run_dbt("deps")

# ---- Step 2: Load seed data (wc2026_fixtures.csv → Postgres) ----
print("\nStep 2: dbt seed (load wc2026_fixtures.csv into Postgres)")
rc_seed, out_seed, err_seed = run_dbt("seed")

# ---- Step 3: Run all models ----
# dbt automatically determines the correct order from the dependency graph
# Order: stg_* → mart_team_form, mart_head_to_head → mart_predictions_input
print("\nStep 3: dbt run (build all staging + mart models)")
rc_run, out_run, err_run = run_dbt("run")

print("\n" + "="*60)
print("dbt run complete. Models should now be in PostgreSQL.")
print("Schemas: 'staging_staging' and 'staging_marts' (or 'public' depending on config)")
print("="*60)

## Running dbt Tests

`dbt test` runs all the tests defined in `schema.yml` files:
- **`not_null`** — fails if any row has NULL in that column
- **`unique`** — fails if any value appears more than once
- **`accepted_values`** — fails if a value not in the allowed list appears

In production (Phase 5), the Airflow DAG is configured to **fail the entire pipeline** if any dbt test fails — preventing bad predictions from reaching the dashboard.

In [ ]:
# ============================================================
# Cell 3 — Run dbt tests and display pass/fail summary
# ============================================================
import re
import json
from pathlib import Path

# ---- Run dbt test ----
print("Running dbt test...")
rc_test, out_test, err_test = run_dbt("test")

# ---- Parse test results ----
# dbt outputs lines like:
#   PASS not_null_stg_match_results_match_date ..... [PASS in 0.12s]
#   FAIL accepted_values_stg_match_results_result ... [FAIL 3]

passed_tests = []
failed_tests = []
warn_tests   = []

for line in out_test.split("\n"):
    line = line.strip()
    if line.startswith("PASS"):
        passed_tests.append(line)
    elif line.startswith("FAIL"):
        failed_tests.append(line)
    elif line.startswith("WARN"):
        warn_tests.append(line)

# ---- Print summary ----
total = len(passed_tests) + len(failed_tests) + len(warn_tests)

print("\n" + "="*60)
print("  DBT TEST RESULTS SUMMARY")
print("="*60)
print(f"  Total tests : {total}")
print(f"  PASSED      : {len(passed_tests)}")
print(f"  FAILED      : {len(failed_tests)}")
print(f"  WARNINGS    : {len(warn_tests)}")
print("="*60)

if failed_tests:
    print("\nFAILED TESTS (investigate these):")
    for t in failed_tests:
        print(f"  {t}")
else:
    print("\nAll tests passed! Data quality is clean.")

if warn_tests:
    print("\nWARNINGS:")
    for t in warn_tests:
        print(f"  {t}")

# ---- Also check using dbt's run results JSON ----
# dbt writes a machine-readable result file to target/run_results.json
run_results_path = Path("..") / "dbt" / "target" / "run_results.json"
if run_results_path.exists():
    with open(run_results_path) as f:
        run_results = json.load(f)
    
    print("\nTest results from target/run_results.json:")
    status_counts = {}
    for result in run_results.get("results", []):
        status = result.get("status", "unknown")
        status_counts[status] = status_counts.get(status, 0) + 1
    for status, count in sorted(status_counts.items()):
        print(f"  {status:<10}: {count}")
else:
    print("\nrun_results.json not found — run 'dbt test' first.")

# ---- Generate docs ----
print("\nGenerating dbt documentation...")
rc_docs, out_docs, _ = run_dbt("docs generate")

if rc_docs == 0:
    print("\nDocs generated! To view:")
    print("  cd ../dbt")
    print("  dbt docs serve --profiles-dir .")
    print("  Then open: http://localhost:8080")

# ---- Final verification: check tables exist in Postgres ----
import pandas as pd
from sqlalchemy import create_engine, text

DB_URL = os.getenv("DATABASE_URL")
if DB_URL:
    engine = create_engine(DB_URL)
    with engine.connect() as conn:
        # List all tables in the database
        tables = conn.execute(text("""
            SELECT table_schema, table_name, 
                   pg_size_pretty(pg_total_relation_size(quote_ident(table_schema)||
                                  '.'||quote_ident(table_name))) as size
            FROM information_schema.tables
            WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
            ORDER BY table_schema, table_name;
        """)).fetchall()
    
    print("\nAll tables in PostgreSQL:")
    print(f"  {'Schema':<20} {'Table':<35} {'Size':>8}")
    print("-" * 65)
    for schema, table, size in tables:
        print(f"  {schema:<20} {table:<35} {size:>8}")
    
    engine.dispose()
else:
    print("DATABASE_URL not set — skipping Postgres table check")